In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

from scipy.ndimage import label, binary_fill_holes
from skimage.morphology import medial_axis

MASKS_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\refined_masks"
CALIBRATION_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\scale_calibration.csv"
SUMMARY_OUTPUT = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
RAW_OUTPUT = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_raw_thickness.csv"
EXCEL_OUTPUT = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_analysis.xlsx"

# Remove very small segmentation noise
MIN_COMPONENT_SIZE = 300

df_scale = pd.read_csv(
    CALIBRATION_CSV
)
scale_map = {}

for _, row in df_scale.iterrows():
    image_name = os.path.splitext(
        str(row["image_name"])
    )[0]
    scale_map[image_name] = float(
        row["nm_per_pixel"]
    )
print(
    "Loaded calibration for",
    len(scale_map),
    "images"
)

#thickness calculation function
def calculate_thickness(mask_path, nm_per_pixel):
    summary_results = []
    raw_results = []

    # Load segmentation mask
    mask = cv2.imread(
        mask_path,
        cv2.IMREAD_GRAYSCALE
    )

    if mask is None:
        return summary_results, raw_results

    binary_mask = mask > 0

    # Fill holes inside membrane
    binary_mask = binary_fill_holes(
        binary_mask
    )

    labelled_mask, number_components = label(
        binary_mask
    )

    # Analyse each membrane
    for component_id in range(
        1,
        number_components + 1
    ):
        component = (
            labelled_mask == component_id
        )

        # Remove noise
        if np.sum(component) < MIN_COMPONENT_SIZE:
            continue

        medial_line, distance_map = medial_axis(
            component,
            return_distance=True
        )

        # Radius -> diameter
        thickness_pixels = (
            distance_map[medial_line] * 2
        )

        # Pixel -> nanometre
        thickness_nm = (
            thickness_pixels * nm_per_pixel
        )
        if len(thickness_nm) == 0:

            continue

        summary_results.append({

            "component_id":
                component_id,

            "mean_thickness_nm":
                np.mean(thickness_nm),

            "median_thickness_nm":
                np.median(thickness_nm),

            "std_thickness_nm":
                np.std(thickness_nm),

            "min_thickness_nm":
                np.min(thickness_nm),

            "max_thickness_nm":
                np.max(thickness_nm),

            "number_of_measurements":
                len(thickness_nm)
        })

        for point, value in enumerate(thickness_nm):
            raw_results.append({
                "component_id":
                    component_id,
                "point_id":
                    point,
                "thickness_nm":
                    value
            })
    return summary_results, raw_results

summary_all = []
raw_all = []

for filename in sorted(
    os.listdir(MASKS_FOLDER)
):
    if filename.lower().endswith(
        (".png", ".jpg", ".tif", ".tiff")
    ):

        base_name = os.path.splitext(
            filename
        )[0]

        mask_path = os.path.join(
            MASKS_FOLDER,
            filename
        )

        # Find scale
        nm_per_pixel = scale_map.get(
            base_name
        )
        if nm_per_pixel is None:

            print(
                "No calibration:",
                filename
            )
            continue

        # Patient ID
        patient_id = base_name.split("_")[0]

        summary, raw = calculate_thickness(
            mask_path,
            nm_per_pixel
        )

        # Add information
        for item in summary:
            item["patient_id"] = patient_id
            item["image_name"] = filename
            item["resolution_nm_pixel"] = nm_per_pixel
            summary_all.append(item)

        for item in raw:
            item["patient_id"] = patient_id
            item["image_name"] = filename
            item["resolution_nm_pixel"] = nm_per_pixel
            raw_all.append(item)

        print(
            filename,
            "completed:",
            len(summary),
            "membranes"
        )

summary_df = pd.DataFrame(
    summary_all
)
raw_df = pd.DataFrame(
    raw_all
)
summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False
)
raw_df.to_csv(
    RAW_OUTPUT,
    index=False
)

with pd.ExcelWriter(
    EXCEL_OUTPUT
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="Raw Thickness",
        index=False
    )

print("GBM thickness analysis finished")
print("="*60)
print(
    "Membrane components:",
    len(summary_df)
)
print(
    "Thickness measurements:",
    len(raw_df)
)
print(
    "Excel file:",
    EXCEL_OUTPUT
)

Loaded calibration for 257 images
01-24_03_1g.png completed: 2 membranes
01-24_04_1g.png completed: 1 membranes
01-24_05_1g.png completed: 1 membranes
01-24_06_1g.png completed: 1 membranes
01-24_07_1g.png completed: 1 membranes
01-24_08_1g.png completed: 1 membranes
01-24_09_1g.png completed: 1 membranes
01-24_10_1g.png completed: 1 membranes
01-24_11_1g.png completed: 1 membranes
01-24_12_1g.png completed: 2 membranes
01-24_13_1g.png completed: 1 membranes
01-24_14_1g.png completed: 1 membranes
01-24_15_1g.png completed: 3 membranes
01-24_16_1g.png completed: 1 membranes
01-24_17_1g.png completed: 1 membranes
02-24_03_2g.png completed: 2 membranes
02-24_04_1g.png completed: 1 membranes
02-24_04_2g.png completed: 1 membranes
02-24_05_1g.png completed: 1 membranes
02-24_05_2g.png completed: 2 membranes
02-24_06_1g.png completed: 1 membranes
02-24_06_2g.png completed: 1 membranes
02-24_07_1g.png completed: 2 membranes
02-24_07_2g.png completed: 1 membranes
02-24_08_1g.png completed: 2 m